# Monte Carlo Pricing of a Phoenix Autocall Note
## GBM and Heston Stochastic Volatility Models

**Product:** 2-Year USD Phoenix Autocall Notes linked to the S&P 500® Index  
**Issuer:** Morgan Stanley B.V. 
**Term:** 2 Years (Strike: 30 Dec 2024, Determination: 30 Dec 2026)

**Models:**
- (i) Geometric Brownian Motion (GBM) — constant volatility
- (ii) Heston Stochastic Volatility

**Monte Carlo Configuration:**
- Paths: 10,000,000
- Time steps: Daily (252 trading days/year × 2 years = 504 steps)
- RNG: NumPy default (PCG-64), seeded for reproducibility

In [9]:
import numpy as np
import time
import sys

## 1. Product Specification (as extracted from Term Sheet)

All contractual parameters are extracted from the term sheet (XS2918085494)

| Feature | Value |
|---------|-------|
| Underlying | S&P 500® Index |
| Denomination (Par) | USD 1,000 |
| Tenor | 2 years |
| Quarterly Coupon | 1.8875% of Par (= USD 18.875) |
| Annualised Coupon | 7.55% |
| Coupon Barrier | 75% of $S_0$ |
| Autocall Level | 100% of $S_0$ |
| Autocall Dates | Q2 – Q7 (no autocall at Q1 or Q8) |
| Final Barrier | 75% of $S_0$ (European) |
| Final Redemption (above) | 100% of Par |
| Final Redemption (below) | Par × ($S_T$ / $S_0$) |
| Memory Coupon | Yes (Phoenix convention) |

**Simplifications Applied:**
1. Equally spaced quarterly observations ($t_n = n/4$ years). Per the brief's recommendation.
2. Observation date = payment date (no settlement lag). Per the brief's permission.
3. Zero dividend yield. Per the brief's permission.

In [10]:
class PhoenixAutocallSpec:
    """
    Term sheet params for the Phoenix Autocall (XS2918085494).
    Memory coupon implemented — standard Phoenix convention.
    Dates simplified to equally spaced quarters, no settlement lag, q=0.
    """
    
    def __init__(self):
        self.par = 1000.0                          # USD denomination
        self.T = 2.0                               # Tenor in years
        self.n_obs = 8                             # Number of quarterly observations
        self.coupon_rate = 0.018875                # Quarterly coupon (1.8875% of Par)
        self.coupon_barrier_pct = 0.75             # Coupon barrier as % of S_0
        self.autocall_level_pct = 1.00             # Autocall level as % of S_0
        self.final_barrier_pct = 0.75              # Final redemption barrier as % of S_0
        
        # Autocall at Q2-Q7 only. No autocall at Q1 or at maturity (Q8)
        self.autocall_obs_indices = [2, 3, 4, 5, 6, 7]
        
        self.coupon_obs_indices = [1, 2, 3, 4, 5, 6, 7, 8]
        
        # Quarterly observation times in years
        self.obs_times = np.array([n / 4.0 for n in range(1, self.n_obs + 1)])
        
    def describe(self):
        """Print a summary of the product specification."""
        print("=" * 72)
        print("PRODUCT SPECIFICATION: 2Y USD Phoenix Autocall — S&P 500")
        print("=" * 72)
        print(f"  Denomination (Par):        USD {self.par:,.0f}")
        print(f"  Tenor:                     {self.T} years")
        print(f"  Quarterly Coupon:          {self.coupon_rate*100:.4f}% of Par"
              f" = USD {self.par * self.coupon_rate:.4f}")
        print(f"  Annualised Coupon:         {self.coupon_rate*4*100:.2f}%")
        print(f"  Coupon Barrier:            {self.coupon_barrier_pct*100:.0f}% of S₀")
        print(f"  Autocall Level:            {self.autocall_level_pct*100:.0f}% of S₀")
        print(f"  Autocall Dates:            Q{self.autocall_obs_indices[0]}"
              f" – Q{self.autocall_obs_indices[-1]}"
              f" (n={self.autocall_obs_indices})")
        print(f"  Final Barrier (European):  {self.final_barrier_pct*100:.0f}% of S₀")
        print(f"  Final Redemption (above):  100% of Par")
        print(f"  Final Redemption (below):  Par × (S_T / S₀)")
        print(f"  Observation Times (yrs):   {self.obs_times}")
        print(f"  Memory Coupon:             Yes (Phoenix convention; see documentation)")
        print("=" * 72)

## 2. Monte Carlo Simulation Engine

Two simulation methods are implemented:

### 2.1 GBM — Exact Log-Normal Discretisation

Under the risk-neutral measure $\mathbb{Q}$:

$$dS = r \, S \, dt + \sigma \, S \, dW$$

Exact solution (zero discretisation bias):

$$\ln S(t+\Delta t) = \ln S(t) + \left(r - \tfrac{1}{2}\sigma^2\right)\Delta t + \sigma\sqrt{\Delta t}\; Z, \quad Z \sim \mathcal{N}(0,1)$$

- $\sigma = 20\%$ (= $\sqrt{v_0}$ for consistency with Heston)
- $r = 4.0\%$ (approximation of 2Y USD Treasury yield, Dec 2024)

### 2.2 Heston Stochastic Volatility — Full Truncation

$$dS = r \, S \, dt + \sqrt{v} \, S \, dW_S$$
$$dv = \kappa(\theta - v)\,dt + \xi\sqrt{v}\, dW_v$$
$$\text{Corr}(dW_S, dW_v) = \rho$$

**Prescribed parameters (no calibration):** $v_0 = 0.04$, $\theta = 0.04$, $\kappa = 1.5$, $\xi = 0.5$, $\rho = -0.7$.

**Feller condition:** $2\kappa\theta / \xi^2 = 0.48 < 1$ (NOT satisfied — variance can reach zero).

**Discretisation:** Euler–Maruyama with full truncation (Lord, Koekkoek & Van Dijk, 2010). Defining $v^+ = \max(v, 0)$:

$$v(t+\Delta t) = v(t) + \kappa(\theta - v^+)\Delta t + \xi\sqrt{v^+}\sqrt{\Delta t}\, Z_v$$
$$\ln S(t+\Delta t) = \ln S(t) + \left(r - \tfrac{1}{2}v^+\right)\Delta t + \sqrt{v^+}\sqrt{\Delta t}\, Z_S$$

**Cholesky decomposition:** $Z_S = \rho\, Z_v + \sqrt{1-\rho^2}\, Z_\perp$

In [11]:
class MonteCarloEngine:
    """
    GBM and Heston simulation engine. Daily steps (dt=1/252), 504 total.
    GBM uses exact log-normal. Heston uses Euler-Maruyama + full truncation.
    Returns index levels at 8 quarterly obs dates only (memory efficiency).
    """
    
    def __init__(self, r=0.04, sigma=0.20, S0=1.0, seed=42):
        self.r = r
        self.sigma = sigma          # GBM vol, set to sqrt(v0) for fair comparison
        self.S0 = S0                # Normalised to 1.0
        self.seed = seed
        
        # Heston parameters 
        self.v0 = 0.04       # Initial variance
        self.theta = 0.04    # Long-run variance
        self.kappa = 1.5     # Mean reversion speed
        self.xi = 0.5        # Vol-of-vol
        self.rho = -0.7      # Correlation (equity-variance)
        
        # Time grid
        self.days_per_year = 252
        self.T = 2.0
        self.n_steps = int(self.T * self.days_per_year)  # 504
        self.dt = 1.0 / self.days_per_year
        
        # Quarterly observation steps: every 63 days
        self.steps_per_quarter = self.days_per_year // 4
        self.obs_step_indices = [
            k * self.steps_per_quarter for k in range(1, 9)
        ]
        
    def simulate_gbm(self, n_paths, batch_size=500_000):
        """GBM paths via exact log-normal. Batched for memory."""
        rng = np.random.default_rng(self.seed)
        obs_levels = np.empty((n_paths, 8), dtype=np.float64)
        
        drift = (self.r - 0.5 * self.sigma**2) * self.dt
        diffusion = self.sigma * np.sqrt(self.dt)
        
        n_batches = (n_paths + batch_size - 1) // batch_size
        
        for b in range(n_batches):
            start = b * batch_size
            end = min(start + batch_size, n_paths)
            bs = end - start
            
            log_S = np.zeros(bs, dtype=np.float64)
            
            obs_idx = 0
            for step in range(1, self.n_steps + 1):
                Z = rng.standard_normal(bs)
                log_S += drift + diffusion * Z
                
                if step == self.obs_step_indices[obs_idx]:
                    obs_levels[start:end, obs_idx] = np.exp(log_S) * self.S0
                    obs_idx += 1
                    if obs_idx >= 8:
                        break
        
        return obs_levels
    
    def simulate_heston(self, n_paths, batch_size=500_000):
        """Heston paths via Euler-Maruyama + full truncation (Lord et al. 2010)."""
        rng = np.random.default_rng(self.seed)
        obs_levels = np.empty((n_paths, 8), dtype=np.float64)
        
        sqrt_dt = np.sqrt(self.dt)
        rho_comp = np.sqrt(1.0 - self.rho**2)
        
        n_batches = (n_paths + batch_size - 1) // batch_size
        
        for b in range(n_batches):
            start = b * batch_size
            end = min(start + batch_size, n_paths)
            bs = end - start
            
            log_S = np.zeros(bs, dtype=np.float64)
            v = np.full(bs, self.v0, dtype=np.float64)
            
            obs_idx = 0
            for step in range(1, self.n_steps + 1):
                # Correlated Brownian increments via Cholesky
                Z_v = rng.standard_normal(bs)
                Z_perp = rng.standard_normal(bs)
                Z_S = self.rho * Z_v + rho_comp * Z_perp
                
                # Full truncation: v+ = max(v, 0)
                v_plus = np.maximum(v, 0.0)
                sqrt_v = np.sqrt(v_plus)
                
                # Log-price update
                log_S += (self.r - 0.5 * v_plus) * self.dt + sqrt_v * sqrt_dt * Z_S
                
                # Variance update
                v = v + self.kappa * (self.theta - v_plus) * self.dt \
                    + self.xi * sqrt_v * sqrt_dt * Z_v
                
                if step == self.obs_step_indices[obs_idx]:
                    obs_levels[start:end, obs_idx] = np.exp(log_S) * self.S0
                    obs_idx += 1
                    if obs_idx >= 8:
                        break
        
        return obs_levels

## 3. Payoff Engine — Memory Coupon Implementation

At each quarterly observation $n = 1, \ldots, 8$:

1. **Coupon check with memory:** If $S(t_n) \geq 0.75 \times S_0$, pay current coupon + all accumulated missed coupons. Otherwise increment the missed counter.

2. **Autocall check** ($n = 2, \ldots, 7$ only): If $S(t_n) \geq 1.00 \times S_0$, pay 100% of Par and terminate. (Coupon already computed in step 1.)

3. **Final maturity** ($n = 8$):
   - Above barrier: 100% of Par (+ coupon/memory from step 1)
   - Below barrier: Par × ($S_T / S_0$), no coupon, missed coupons forfeited

All cash flows discounted at $D(t) = e^{-rt}$.

In [12]:
class PhoenixPayoffEngine:
    """
    Payoff engine with memory coupon. Processes each quarter:
    coupon check -> autocall check -> final redemption. All discounted at r.
    """
    
    def __init__(self, spec, r):
        self.spec = spec
        self.r = r
        self.discount_factors = np.exp(-r * spec.obs_times)
        
    def compute_payoffs(self, obs_levels):
        """Vectorised payoff computation. Returns (pv array, stats dict)."""
        n_paths = obs_levels.shape[0]
        spec = self.spec
        S0 = 1.0
        
        coupon_barrier = spec.coupon_barrier_pct * S0
        autocall_level = spec.autocall_level_pct * S0
        final_barrier = spec.final_barrier_pct * S0
        coupon_amount = spec.coupon_rate * spec.par
        
        pv = np.zeros(n_paths, dtype=np.float64)
        alive = np.ones(n_paths, dtype=bool)
        missed_coupons = np.zeros(n_paths, dtype=np.int64)
        
        # Diagnostics
        autocall_counts = np.zeros(8, dtype=np.int64)
        coupon_counts = np.zeros(8, dtype=np.int64)
        memory_coupons_paid = np.zeros(8, dtype=np.int64)
        final_above_barrier = 0
        final_below_barrier = 0
        
        for k in range(8):
            n = k + 1
            S_k = obs_levels[:, k]
            df_k = self.discount_factors[k]
            
            # Coupon check with memory
            barrier_met = alive & (S_k >= coupon_barrier)
            barrier_not_met = alive & (S_k < coupon_barrier)
            
            total_coupons = 1 + missed_coupons[barrier_met]
            pv[barrier_met] += coupon_amount * total_coupons * df_k
            
            coupon_counts[k] = np.sum(barrier_met)
            memory_coupons_paid[k] = int(np.sum(missed_coupons[barrier_met]))
            
            missed_coupons[barrier_met] = 0
            missed_coupons[barrier_not_met] += 1
            
            # Autocall check (Q2-Q7 only)
            if n in spec.autocall_obs_indices:
                autocalled = alive & (S_k >= autocall_level)
                pv[autocalled] += spec.par * df_k
                autocall_counts[k] = np.sum(autocalled)
                alive[autocalled] = False
            
            # Final maturity
            if n == 8:
                above = alive & (S_k >= final_barrier)
                below = alive & (S_k < final_barrier)
                
                pv[above] += spec.par * df_k
                pv[below] += spec.par * (S_k[below] / S0) * df_k
                
                final_above_barrier = int(np.sum(above))
                final_below_barrier = int(np.sum(below))
        
        total_autocalled = int(np.sum(~alive))
        total_memory_paid = int(np.sum(memory_coupons_paid))
        stats = {
            'n_paths': n_paths,
            'autocall_counts_by_quarter': autocall_counts.tolist(),
            'total_autocalled': total_autocalled,
            'autocall_probability': total_autocalled / n_paths,
            'coupon_counts_by_quarter': coupon_counts.tolist(),
            'memory_coupons_paid_by_quarter': memory_coupons_paid.tolist(),
            'total_memory_coupons_paid': total_memory_paid,
            'avg_coupons_per_path': sum(coupon_counts) / n_paths,
            'final_above_barrier': final_above_barrier,
            'final_below_barrier': final_below_barrier,
            'final_above_barrier_pct': final_above_barrier / n_paths * 100,
            'final_below_barrier_pct': final_below_barrier / n_paths * 100,
            'reached_maturity_pct': (final_above_barrier + final_below_barrier) / n_paths * 100,
        }
        
        return pv, stats

## 4. Modelling Assumptions

| # | Assumption | Justification |
|---|-----------|---------------|
| A1 | Risk-neutral pricing framework | Fair value = $\mathbb{E}_\mathbb{Q}$[discounted payoff] |
| A2 | Constant risk-free rate $r = 4.0\%$ | Approximates 2Y USD Treasury yield (Dec 2024) |
| A3 | Zero dividend yield | Permitted by the project brief |
| A4 | GBM volatility $\sigma = 20\%$ | = $\sqrt{v_0}$ for consistency with Heston |
| A5 | Heston parameters as prescribed | No calibration performed |
| A6 | Normalised $S_0 = 1.0$ | Barriers expressed as fractions directly |
| A7 | Equally spaced quarterly observations | Per the brief's recommendation |
| A8 | Observation date = payment date | No settlement lag (per brief) |
| A9 | Memory coupon (Phoenix convention) | Missed coupons accumulate and pay on next barrier hit |
| A10 | European barrier for final redemption | Observed only at Determination Date |
| A11 | Daily time stepping ($\Delta t = 1/252$) | 504 total steps over 2-year horizon |
| A12 | No variance reduction | Standard pseudo-random MC, 10M paths |
| A13 | Heston: Euler–Maruyama, full truncation | Lord et al. (2010); Feller condition violated |
| A14 | GBM: exact log-normal discretisation | Zero discretisation bias |
| A15 | Reproducibility | NumPy PCG-64, seed = 42 |

## 5. Configuration & Initialisation

In [13]:
#  Configuration 
N_PATHS        = 10_000_000
RISK_FREE_RATE = 0.04
GBM_SIGMA      = 0.20       # = sqrt(v0) for consistency
SEED           = 42

# Initialise components 
spec          = PhoenixAutocallSpec()
engine        = MonteCarloEngine(r=RISK_FREE_RATE, sigma=GBM_SIGMA, S0=1.0, seed=SEED)
payoff_engine = PhoenixPayoffEngine(spec, r=RISK_FREE_RATE)

spec.describe()

print(f"\nMonte Carlo Configuration:")
print(f"  Paths:             {N_PATHS:>14,}")
print(f"  Daily time steps:  {engine.n_steps:>14}")
print(f"  dt:                {engine.dt:>14.6f} years")
print(f"  Risk-free rate:    {RISK_FREE_RATE*100:>13.1f}%")
print(f"  GBM volatility:    {GBM_SIGMA*100:>13.1f}%")
print(f"  RNG seed:          {SEED:>14}")
print(f"  Batch size:        {'500,000':>14}")

PRODUCT SPECIFICATION: 2Y USD Phoenix Autocall — S&P 500
  Denomination (Par):        USD 1,000
  Tenor:                     2.0 years
  Quarterly Coupon:          1.8875% of Par = USD 18.8750
  Annualised Coupon:         7.55%
  Coupon Barrier:            75% of S₀
  Autocall Level:            100% of S₀
  Autocall Dates:            Q2 – Q7 (n=[2, 3, 4, 5, 6, 7])
  Final Barrier (European):  75% of S₀
  Final Redemption (above):  100% of Par
  Final Redemption (below):  Par × (S_T / S₀)
  Observation Times (yrs):   [0.25 0.5  0.75 1.   1.25 1.5  1.75 2.  ]
  Memory Coupon:             Yes (Phoenix convention; see documentation)

Monte Carlo Configuration:
  Paths:                 10,000,000
  Daily time steps:             504
  dt:                      0.003968 years
  Risk-free rate:              4.0%
  GBM volatility:             20.0%
  RNG seed:                      42
  Batch size:               500,000


## 6. Model 1 — Geometric Brownian Motion (GBM)

In [14]:
print("=" * 72)
print("MODEL 1: GEOMETRIC BROWNIAN MOTION (GBM)")
print("=" * 72)
print("  Dynamics: dS = r·S·dt + σ·S·dW")
print(f"  Parameters: r = {RISK_FREE_RATE}, σ = {GBM_SIGMA}")
print(f"  Discretisation: Exact log-normal (no bias)")
print(f"\n  Simulating {N_PATHS:,} paths")

t0 = time.time()
gbm_obs = engine.simulate_gbm(N_PATHS)
t_gbm_sim = time.time() - t0
print(f"  Simulation time: {t_gbm_sim:.1f}s")

print("  Computing payoffs")
t0 = time.time()
gbm_pv, gbm_stats = payoff_engine.compute_payoffs(gbm_obs)
t_gbm_pay = time.time() - t0
print(f"  Payoff computation time: {t_gbm_pay:.1f}s")

gbm_price = np.mean(gbm_pv)
gbm_se    = np.std(gbm_pv, ddof=1) / np.sqrt(N_PATHS)
gbm_ci_lo = gbm_price - 1.96 * gbm_se
gbm_ci_hi = gbm_price + 1.96 * gbm_se

print(f"\n  GBM Fair Value: USD {gbm_price:.4f} ± {gbm_se:.4f}")
print(f"  95% CI: [{gbm_ci_lo:.4f}, {gbm_ci_hi:.4f}]")

MODEL 1: GEOMETRIC BROWNIAN MOTION (GBM)
  Dynamics: dS = r·S·dt + σ·S·dW
  Parameters: r = 0.04, σ = 0.2
  Discretisation: Exact log-normal (no bias)

  Simulating 10,000,000 paths
  Simulation time: 117.7s
  Computing payoffs
  Payoff computation time: 4.9s

  GBM Fair Value: USD 998.3612 ± 0.0333
  95% CI: [998.2960, 998.4265]


## 7. Model 2 — Heston Stochastic Volatility

In [15]:
print("=" * 72)
print("MODEL 2: HESTON STOCHASTIC VOLATILITY")
print("=" * 72)
print("  Dynamics: dS = r·S·dt + √v·S·dW_S")
print("           dv = κ(θ - v)dt + ξ√v·dW_v")
print(f"  Parameters: v₀={engine.v0}, θ={engine.theta}, κ={engine.kappa},"
      f" ξ={engine.xi}, ρ={engine.rho}")
feller = 2 * engine.kappa * engine.theta / engine.xi**2
print(f"  Feller condition: 2κθ/ξ² = {feller:.2f}"
      f" {'≥ 1 (satisfied)' if feller >= 1 else '< 1 (NOT satisfied — variance can reach zero)'}")
print(f"  Discretisation: Euler-Maruyama with full truncation")
print(f"\n  Simulating {N_PATHS:,} paths")

t0 = time.time()
heston_obs = engine.simulate_heston(N_PATHS)
t_heston_sim = time.time() - t0
print(f"  Simulation time: {t_heston_sim:.1f}s")

print("  Computing payoffs")
t0 = time.time()
heston_pv, heston_stats = payoff_engine.compute_payoffs(heston_obs)
t_heston_pay = time.time() - t0
print(f"  Payoff computation time: {t_heston_pay:.1f}s")

heston_price = np.mean(heston_pv)
heston_se    = np.std(heston_pv, ddof=1) / np.sqrt(N_PATHS)
heston_ci_lo = heston_price - 1.96 * heston_se
heston_ci_hi = heston_price + 1.96 * heston_se

print(f"\n  Heston Fair Value: USD {heston_price:.4f} ± {heston_se:.4f}")
print(f"  95% CI: [{heston_ci_lo:.4f}, {heston_ci_hi:.4f}]")

MODEL 2: HESTON STOCHASTIC VOLATILITY
  Dynamics: dS = r·S·dt + √v·S·dW_S
           dv = κ(θ - v)dt + ξ√v·dW_v
  Parameters: v₀=0.04, θ=0.04, κ=1.5, ξ=0.5, ρ=-0.7
  Feller condition: 2κθ/ξ² = 0.48 < 1 (NOT satisfied — variance can reach zero)
  Discretisation: Euler-Maruyama with full truncation

  Simulating 10,000,000 paths
  Simulation time: 439.8s
  Computing payoffs
  Payoff computation time: 4.4s

  Heston Fair Value: USD 993.3517 ± 0.0397
  95% CI: [993.2738, 993.4296]


## 8. Pricing Results

In [16]:

# RESULTS REPORT

print("\n" + "=" * 72)
print("PRICING RESULTS")
print("=" * 72)

print(f"\n{'':>30} {'GBM':>16} {'Heston':>16}")
print(f"  {'─' * 60}")
print(f"  {'Fair Value (USD)':>28} {gbm_price:>16.4f} {heston_price:>16.4f}")
print(f"  {'Std Error':>28} {gbm_se:>16.4f} {heston_se:>16.4f}")
print(f"  {'95% CI Lower':>28} {gbm_ci_lo:>16.4f} {heston_ci_lo:>16.4f}")
print(f"  {'95% CI Upper':>28} {gbm_ci_hi:>16.4f} {heston_ci_hi:>16.4f}")
print(f"  {'Fair Value (% of Par)':>28} {gbm_price/spec.par*100:>15.2f}% {heston_price/spec.par*100:>15.2f}%")
print(f"  {'Difference (Heston - GBM)':>28} {(heston_price - gbm_price):>16.4f} USD")
print(f"  {'Difference (bps of Par)':>28} {(heston_price - gbm_price)/spec.par*10000:>14.1f} bps")

print(f"\n{'':>30} {'GBM':>16} {'Heston':>16}")
print(f"  {'─' * 60}")
print(f"  {'Autocall Probability':>28} {gbm_stats['autocall_probability']*100:>15.2f}% "
      f"{heston_stats['autocall_probability']*100:>15.2f}%")
print(f"  {'Avg Coupons/Path':>28} {gbm_stats['avg_coupons_per_path']:>16.3f} "
      f"{heston_stats['avg_coupons_per_path']:>16.3f}")
print(f"  {'Total Memory Coupons Paid':>28} {gbm_stats['total_memory_coupons_paid']:>14,} "
      f"{heston_stats['total_memory_coupons_paid']:>14,}")
print(f"  {'Reached Maturity':>28} {gbm_stats['reached_maturity_pct']:>15.2f}% "
      f"{heston_stats['reached_maturity_pct']:>15.2f}%")
print(f"  {'Final: Above Barrier':>28} {gbm_stats['final_above_barrier_pct']:>15.2f}% "
      f"{heston_stats['final_above_barrier_pct']:>15.2f}%")
print(f"  {'Final: Below Barrier':>28} {gbm_stats['final_below_barrier_pct']:>15.2f}% "
      f"{heston_stats['final_below_barrier_pct']:>15.2f}%")

# Per-quarter breakdown
print(f"\n  Autocall Counts by Quarter:")
print(f"  {'Quarter':>10} {'GBM':>14} {'Heston':>14} {'GBM %':>10} {'Heston %':>10}")
for k in range(8):
    n = k + 1
    gc = gbm_stats['autocall_counts_by_quarter'][k]
    hc = heston_stats['autocall_counts_by_quarter'][k]
    marker = " (no AC)" if n not in spec.autocall_obs_indices else ""
    print(f"  {'Q'+str(n):>10} {gc:>14,} {hc:>14,} "
          f"{gc/N_PATHS*100:>9.2f}% {hc/N_PATHS*100:>9.2f}%{marker}")

print(f"\n  Coupon Counts by Quarter:")
print(f"  {'Quarter':>10} {'GBM':>14} {'Heston':>14} {'GBM %':>10} {'Heston %':>10}")
for k in range(8):
    n = k + 1
    gc = gbm_stats['coupon_counts_by_quarter'][k]
    hc = heston_stats['coupon_counts_by_quarter'][k]
    print(f"  {'Q'+str(n):>10} {gc:>14,} {hc:>14,} "
          f"{gc/N_PATHS*100:>9.2f}% {hc/N_PATHS*100:>9.2f}%")

print(f"\n  Memory Coupons Paid by Quarter (accumulated missed coupons recovered):")
print(f"  {'Quarter':>10} {'GBM':>14} {'Heston':>14}")
for k in range(8):
    n = k + 1
    gc = gbm_stats['memory_coupons_paid_by_quarter'][k]
    hc = heston_stats['memory_coupons_paid_by_quarter'][k]
    print(f"  {'Q'+str(n):>10} {gc:>14,} {hc:>14,}")

# Distributional summary
print(f"\n  Payoff Distribution Summary:")
for label, pv_arr in [("GBM", gbm_pv), ("Heston", heston_pv)]:
    print(f"\n  {label}:")
    print(f"    Mean:      {np.mean(pv_arr):>10.4f}")
    print(f"    Median: {np.median(pv_arr):>10.4f}")
    print(f"    Std:    {np.std(pv_arr):>10.4f}")
    print(f"    Min:    {np.min(pv_arr):>10.4f}")
    print(f"    Max:    {np.max(pv_arr):>10.4f}")
    pcts = [1, 5, 10, 25, 75, 90, 95, 99]
    vals = np.percentile(pv_arr, pcts)
    for p, v in zip(pcts, vals):
        print(f"    P{p:<3d}:   {v:>10.4f}")



PRICING RESULTS

                                            GBM           Heston
  ────────────────────────────────────────────────────────────
              Fair Value (USD)         998.3612         993.3517
                     Std Error           0.0333           0.0397
                  95% CI Lower         998.2960         993.2738
                  95% CI Upper         998.4265         993.4296
         Fair Value (% of Par)           99.84%           99.34%
     Difference (Heston - GBM)          -5.0095 USD
       Difference (bps of Par)          -50.1 bps

                                            GBM           Heston
  ────────────────────────────────────────────────────────────
          Autocall Probability           77.31%           82.80%
              Avg Coupons/Path            3.455            2.963
     Total Memory Coupons Paid      1,431,159      1,416,289
              Reached Maturity           22.69%           17.20%
          Final: Above Barrier           1